In [1]:
import pandas as pd
import numpy as np

print("=" * 60)
print("Excel Data Transformation Script - TEST_DATA ONLY")
print("=" * 60)

# Read Excel file
print("\nReading Excel file...")
test_data = pd.read_excel(base_dir / "test_cleaned_no_outliers_k3.xlsx")

Excel Data Transformation Script - TEST_DATA ONLY

Reading Excel file...
TEST_DATA shape: (1048575, 18)


In [2]:
# STEP 1: Remove 'Civics' subject rows from TEST_DATA
print("\n" + "="*60)
print("STEP 1: Removing 'Civics' subject rows from TEST_DATA")
print("="*60)

if 'EXAM_SUBJECT' in test_data.columns:
    original_rows = len(test_data)
    print(f"\nOriginal rows: {original_rows}")
    print(f"Unique subjects before removal:")
    print(test_data['EXAM_SUBJECT'].value_counts(dropna=False))
    
    test_data = test_data[test_data['EXAM_SUBJECT'] != 'CIVICS']
    removed_rows = original_rows - len(test_data)
    
    print(f"\nRemoved {removed_rows} rows with 'CIVICS' subject")
    print(f"Remaining rows: {len(test_data)}")
    print(f"\nUnique subjects after removal:")
    print(test_data['EXAM_SUBJECT'].value_counts(dropna=False))
else:
    print("EXAM_SUBJECT column not found")


STEP 1: Removing 'Civics' subject rows from TEST_DATA

Original rows: 1048575
Unique subjects before removal:
EXAM_SUBJECT
REASONING         278233
SOCIAL_STUDIES    266801
MATH              265089
SCIENCE           231399
CIVICS              6168
NaN                  885
Name: count, dtype: int64

Removed 6168 rows with 'CIVICS' subject
Remaining rows: 1042407

Unique subjects after removal:
EXAM_SUBJECT
REASONING         278233
SOCIAL_STUDIES    266801
MATH              265089
SCIENCE           231399
NaN                  885
Name: count, dtype: int64


In [3]:
# STEP 2: Transform qualitative data to quantitative
print("\n" + "="*60)
print("STEP 2: Transforming qualitative to quantitative data")
print("="*60)

# LANGUAGE_CODE (TEST_DATA)
print("\n--- LANGUAGE_CODE (TEST_DATA) ---")
if 'LANGUAGE_CODE' in test_data.columns:
    print(f"Unique values: {test_data['LANGUAGE_CODE'].unique()}")
    print(f"Value counts:")
    print(test_data['LANGUAGE_CODE'].value_counts(dropna=False))
    
    language_mapping = {
        'ENU': 1,  # English
        'ESP': 2   # Spanish
    }
    
    test_data['LANGUAGE_CODE_NUMERIC'] = test_data['LANGUAGE_CODE'].map(language_mapping)
    # Blanks remain as NaN
    
    print("✓ LANGUAGE_CODE transformed")
    print(f"Mapping preview:")
    print(test_data[['LANGUAGE_CODE', 'LANGUAGE_CODE_NUMERIC']].drop_duplicates().sort_values('LANGUAGE_CODE_NUMERIC'))
else:
    print("LANGUAGE_CODE column not found")


STEP 2: Transforming qualitative to quantitative data

--- LANGUAGE_CODE (TEST_DATA) ---
Unique values: ['ENU' 'ESP']
Value counts:
LANGUAGE_CODE
ENU    994314
ESP     48093
Name: count, dtype: int64
✓ LANGUAGE_CODE transformed
Mapping preview:
  LANGUAGE_CODE  LANGUAGE_CODE_NUMERIC
0           ENU                      1
9           ESP                      2


In [8]:
# EXAM_SUBJECT (TEST_DATA)
print("\n--- EXAM_SUBJECT (TEST_DATA) ---")
if 'EXAM_SUBJECT' in test_data.columns:
    print(f"Unique values: {test_data['EXAM_SUBJECT'].unique()}")
    print(f"Value counts:")
    print(test_data['EXAM_SUBJECT'].value_counts(dropna=False))
    
    subject_mapping = {
        'MATH': 1,
        'SCIENCE': 2,
        'REASONING': 3,
        'SOCIAL_STUDIES': 4,
        'NULL': 'NULL'  # Keep NULL as NULL
    }
    
    # For NULL values, we need special handling
    test_data['EXAM_SUBJECT_NUMERIC'] = test_data['EXAM_SUBJECT'].apply(
        lambda x: 'NULL' if x == 'NULL' else subject_mapping.get(x, x)
    )
    
    print("✓ EXAM_SUBJECT transformed")
    print(f"Mapping preview:")
    print(test_data[['EXAM_SUBJECT', 'EXAM_SUBJECT_NUMERIC']].drop_duplicates().sort_values(
        'EXAM_SUBJECT_NUMERIC', 
        key=lambda col: col.apply(lambda x: 999 if x == 'NULL' else (x if isinstance(x, (int, float)) else 0))
    ))
else:
    print("EXAM_SUBJECT column not found")


--- EXAM_SUBJECT (TEST_DATA) ---
Unique values: ['SOCIAL_STUDIES' 'MATH' 'SCIENCE' 'REASONING' nan]
Value counts:
EXAM_SUBJECT
REASONING         278233
SOCIAL_STUDIES    266801
MATH              265089
SCIENCE           231399
NaN                  885
Name: count, dtype: int64
✓ EXAM_SUBJECT transformed
Mapping preview:
        EXAM_SUBJECT  EXAM_SUBJECT_NUMERIC
1               MATH                   1.0
3            SCIENCE                   2.0
5          REASONING                   3.0
0     SOCIAL_STUDIES                   4.0
6019             NaN                   NaN


In [5]:
# STEP 3: Save transformed data
print("\n" + "="*60)
print("STEP 3: Saving transformed data")
print("="*60)

output_file = 'C:/Users/davey/OneDrive/Desktop/dbo.TEST_DATA_TRANSFORMED_QUAL_2_QUANT.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    test_data.to_excel(writer, sheet_name='TEST_DATA', index=False)

print(f"\n✓ Transformed data saved to '{output_file}'")


STEP 3: Saving transformed data

✓ Transformed data saved to 'C:/Users/davey/OneDrive/Desktop/dbo.TEST_DATA_TRANSFORMED_QUAL_2_QUANT.xlsx'


In [6]:
# Summary
print("\n" + "="*60)
print("TRANSFORMATION SUMMARY")
print("="*60)

print(f"\nTEST_DATA:")
print(f"  Total rows: {len(test_data):,}")
print(f"  Total columns: {len(test_data.columns)}")
numeric_cols_test = [col for col in test_data.columns if '_NUMERIC' in col]
print(f"  New numeric columns: {len(numeric_cols_test)}")
if numeric_cols_test:
    for col in numeric_cols_test:
        print(f"    - {col}")

print("\n" + "="*60)
print("✓ All transformations completed successfully!")
print("="*60)


TRANSFORMATION SUMMARY

TEST_DATA:
  Total rows: 1,042,407
  Total columns: 20
  New numeric columns: 2
    - LANGUAGE_CODE_NUMERIC
    - EXAM_SUBJECT_NUMERIC

✓ All transformations completed successfully!


In [7]:
# Display sample of transformed data
print("\n" + "="*60)
print("SAMPLE OF TRANSFORMED DATA")
print("="*60)

print("\nTEST_DATA (first 10 rows):")
display_cols = ['LANGUAGE_CODE', 'LANGUAGE_CODE_NUMERIC', 'EXAM_SUBJECT', 'EXAM_SUBJECT_NUMERIC']
display_cols_available = [col for col in display_cols if col in test_data.columns]
if display_cols_available:
    print(test_data[display_cols_available].head(10))
else:
    print(test_data.head(10))


SAMPLE OF TRANSFORMED DATA

TEST_DATA (first 10 rows):
  LANGUAGE_CODE  LANGUAGE_CODE_NUMERIC    EXAM_SUBJECT EXAM_SUBJECT_NUMERIC
0           ENU                      1  SOCIAL_STUDIES       SOCIAL_STUDIES
1           ENU                      1            MATH                    1
2           ENU                      1  SOCIAL_STUDIES       SOCIAL_STUDIES
3           ENU                      1         SCIENCE                    2
4           ENU                      1  SOCIAL_STUDIES       SOCIAL_STUDIES
5           ENU                      1       REASONING                    3
6           ENU                      1  SOCIAL_STUDIES       SOCIAL_STUDIES
7           ENU                      1  SOCIAL_STUDIES       SOCIAL_STUDIES
8           ENU                      1            MATH                    1
9           ESP                      2  SOCIAL_STUDIES       SOCIAL_STUDIES
